# 01 · Preparació de les dades

**Cas pràctic (secció 7 de l'informe)** — dataset públic *Telco Customer Churn* (IBM).

Aquest notebook fixa les decisions de preparació de les dades per a l'anàlisi de supervivència.
Tot el codi és a `src/preparacio.py`, de manera que els notebooks i el dashboard treballen
exactament amb els mateixos clients i la mateixa partició *train/test*.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.precision", 3)

In [2]:
from src.preparacio import carregar_dades, codificar_cox, dividir_train_test, REFERENCIES
bruts = pd.read_csv("../data/Telco-Customer-Churn.csv")
df = carregar_dades()
print(f"Registres originals: {len(bruts)}  ·  després de netejar: {len(df)}")

Registres originals: 7043  ·  després de netejar: 7032


## Definició de l'esdeveniment i de la censura

| Concepte | Definició operativa |
|---|---|
| Temps (`duration`) | `tenure`: mesos d'antiguitat del client |
| Esdeveniment (`event = 1`) | `Churn = "Yes"`: el client ha marxat |
| Censura per la dreta (`event = 0`) | `Churn = "No"`: el client continua actiu en el moment de l'observació; només sabem que ha sobreviscut *com a mínim* `tenure` mesos |

Els supòsits i biaixos d'aquesta interpretació (data de tall desconeguda, possible truncament
esquerre, biaix de selecció…) estan documentats a `docs/informe_metodologic_supervivencia.md`.

In [3]:
resum = pd.DataFrame({
    "clients": [len(df)],
    "baixes (esdeveniments)": [int(df.event.sum())],
    "censurats": [int((df.event == 0).sum())],
    "% censura": [f"{(df.event == 0).mean():.1%}"],
    "antiguitat mín–màx": [f"{df.duration.min()}–{df.duration.max()} mesos"],
})
resum

,clients,baixes (esdeveniments),censurats,% censura,antiguitat mín–màx
0,7032,1869,5163,73.4%,1–72 mesos


## Decisions de neteja

1. **Clients amb `tenure = 0` (11 registres) → exclosos.** Són altes del mateix mes, tots amb
   `Churn = "No"` i `TotalCharges` buit: no aporten temps en risc observat.
2. **`TotalCharges` → numèric**, però **no s'usa als models de supervivència**: és aproximadament
   `tenure × MonthlyCharges`, per tant conté la mateixa informació que la variable resposta
   (*data leakage*).
3. **"No internet service" → "No"** a `OnlineSecurity` i `TechSupport`: si no, aquestes categories
   serien idèntiques a `InternetService = No` (col·linealitat perfecta al model de Cox).

In [4]:
bruts[bruts["tenure"] == 0][["customerID", "tenure", "TotalCharges", "Churn"]]

,customerID,tenure,TotalCharges,Churn
488,4472-LVYGI,0,,No
753,3115-CZMZD,0,,No
936,5709-LVOEQ,0,,No
1082,4367-NUYAO,0,,No
1340,1371-DWPAZ,0,,No
3331,7644-OMVMY,0,,No
3826,3213-VVOLG,0,,No
4380,2520-SGTTA,0,,No
5218,2923-ARZLG,0,,No
6670,4075-WKNIU,0,,No


## Codificació per al model de Cox

Variables categòriques codificades com a *dummies*; la categoria de referència (sense dummy) és:

In [5]:
pd.DataFrame(REFERENCIES.items(), columns=["variable", "categoria de referència"])

,variable,categoria de referència
0,Contract,Month-to-month
1,InternetService,DSL
2,PaymentMethod,Mailed check
3,OnlineSecurity,No
4,TechSupport,No
5,PaperlessBilling,No
6,Partner,No
7,Dependents,No


In [6]:
X = codificar_cox(df)
X.head()

,duration,event,MonthlyCharges,SeniorCitizen,Contract_One year,Contract_Two year,InternetService_Fiber optic,InternetService_No,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,OnlineSecurity_Yes,TechSupport_Yes,PaperlessBilling_Yes,Partner_Yes,Dependents_Yes
customerID,,,,,,,,,,,,,,,,
7590-VHVEG,1,0,29.85,0,0,0,0,0,0,0,1,0,0,1,1,0
5575-GNVDE,34,0,56.95,0,1,0,0,0,0,0,0,1,0,0,0,0
3668-QPYBK,2,1,53.85,0,0,0,0,0,0,0,0,1,0,1,0,0
7795-CFOCW,45,0,42.30,0,1,0,0,0,1,0,0,1,1,0,0,0
9237-HQITU,2,1,70.70,0,0,0,1,0,0,0,1,0,0,1,0,0


## Partició train / test

80 % / 20 %, estratificada per l'esdeveniment, `random_state = 42`. El model de Cox s'ajusta amb
*train* i la seva capacitat predictiva (C-index) s'avalua amb *test*.

In [7]:
tr, te = dividir_train_test(df)
pd.DataFrame({"conjunt": ["train", "test"], "clients": [len(tr), len(te)],
              "taxa de baixa": [tr.event.mean(), te.event.mean()]})

,conjunt,clients,taxa de baixa
0,train,5625,0.266
1,test,1407,0.266


## Conclusions

- Base d'anàlisi: **7.032 clients**, **1.869 baixes** (26,6 %) i **73,4 % d'observacions censurades**.
- Una taxa de censura tan alta és precisament el motiu pel qual cal l'anàlisi de supervivència:
  un classificador que tracti "no ha marxat encara" com "no marxarà" perd la informació temporal.
- Partició reproduïble: el C-index de test mesura la capacitat predictiva sobre clients no vistos.